### Test the TOP 15 models on sub-p0002

In [ ]:
import json 
from pathlib import Path
import numpy as np
import pandas as pd

from nilearn.image import load_img, index_img, resample_to_img, new_img_like
from nilearn.maskers import NiftiMasker
from nilearn.datasets import fetch_atlas_destrieux_2009
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

from torch import FloatTensor, LongTensor, no_grad, max
from torch.nn import Module, Linear, ReLU, Sequential, Dropout

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand': ['E4', 'E5', 'E6', 'E7'],
    'Forearm': ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm': ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20']
}

ELECTRODE_TO_REGION = {
    electrode: region 
    for region, electrodes in REGION_TO_ELECTRODES.items() 
    for electrode in electrodes
}

REGION_TO_LABEL = {
    'Middle_Finger': 0,
    'Hand': 1,
    'Forearm': 2,
    'Arm': 3
}

LABEL_TO_REGION = {v: k for k, v in REGION_TO_LABEL.items()}

print("Region-to-Electrode Mapping:")
for region, electrodes in REGION_TO_ELECTRODES.items():
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {electrodes} ({len(electrodes)} electrodes)")

In [ ]:
class SomatotopicANN(Module):
    def __init__(self, input_size, hidden1, hidden2=None, num_classes=4, dropout_rate=0.3):
        super(SomatotopicANN, self).__init__()
        
        layers = []
        
        layers.append(Linear(input_size, hidden1))
        layers.append(ReLU())
        layers.append(Dropout(dropout_rate))
        
        if hidden2 is not None:
            layers.append(Linear(hidden1, hidden2))
            layers.append(ReLU())
            layers.append(Dropout(dropout_rate))
            layers.append(Linear(hidden2, num_classes))
        else:
            layers.append(Linear(hidden1, num_classes))
        
        self.network = Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

def calculate_specificity(y_true, y_pred, class_label):
    y_true_binary = (y_true == class_label).astype(int)
    y_pred_binary = (y_pred == class_label).astype(int)
    tn = np.sum((y_true_binary == 0) & (y_pred_binary == 0))
    fp = np.sum((y_true_binary == 0) & (y_pred_binary == 1))
    return tn / (tn + fp) if (tn + fp) > 0 else 0.0

In [ ]:
BIDS_ROOT = Path(r"../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"

subject = "sub-p0002"
session = "ses-01"
task = "task-S1Map"
space = "MNI152NLin2009cAsym"
n_runs = 4

HRF_DELAY = 5.0
WINDOW = 1

bold_json = BIDS_ROOT / subject / session / "func" / f"{subject}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])

print(f"Subject: {subject}")
print(f"Runs: {n_runs}")
print(f"TR: {tr} s")
print(f"HRF delay: {HRF_DELAY} s")
print(f"Window: {WINDOW} volumes")

TRAIN_RESULTS_DIR = Path("results/4_Classes_ANN")
TEST_RESULTS_DIR = Path("results/4_Classes_ANN/sub-p0002_testing")
TEST_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
all_events = []
for run in range(1, n_runs + 1):
    events_path = BIDS_ROOT / subject / session / "func" / f"{subject}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(events_path, sep='\t')
    df['subject'] = subject
    df['run'] = run
    all_events.append(df)

events_df = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)

print(f"Total events: {len(events_df)}")
print(f"Stimulation events: {len(stim_events)}")
print(f"Unique electrodes: {stim_events['trial_type'].nunique()}")
print(f"Unique regions: {stim_events['region'].nunique()}")
print(f"\nSamples per region:")
region_counts = stim_events['region'].value_counts()
for region in ['Middle_Finger', 'Hand', 'Forearm', 'Arm']:
    count = region_counts.get(region, 0)
    print(f"  {region} (Class {REGION_TO_LABEL[region]}): {count} samples")

In [ ]:
first_run_path = (DERIVATIVES / subject / session / "func" /
                  f"{subject}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii.gz")
first_run_img = load_img(str(first_run_path))
ref_img = index_img(first_run_img, 0)
print(f"Reference image shape: {first_run_img.shape}")

atlas = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
mask_data = np.isin(atlas_data, s1_indices).astype('uint8')
s1_mask = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')

print(f"Number of voxels in original S1 mask: {int(np.sum(mask_data))}")
print(f"Number of voxels in resampled S1 mask: {int(np.sum(s1_mask_resampled.get_fdata() > 0))}")

masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)

print(f"\nS1 atlas indices: {s1_indices}")
print('Selected atlas regions:')
for i in s1_indices:
    print(f"  - {atlas.labels[i]}")

In [ ]:
X_list = []
y_list = []
run_labels = []

for run in range(1, n_runs + 1):
    bold_path = (DERIVATIVES / subject / session / "func" /
                  f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii.gz")
    bold_img = load_img(str(bold_path))
    print(f"Run {run}...", end=' ')
    
    run_events = stim_events[stim_events['run'] == run]
    for _, event in run_events.iterrows():
        onset = event["onset"]
        trial_type = event["trial_type"]
        region = event["region"]
        target_time = onset + HRF_DELAY
        vol_indx = int(np.round(target_time / tr))
        
        if vol_indx < bold_img.shape[3]:
            vol_img = index_img(bold_img, vol_indx)
            vol_data = masker.transform(vol_img)
            if len(vol_data.shape) == 2:
                X_list.append(vol_data[0])
            else:
                X_list.append(vol_data)
            y_list.append(region) 
            run_labels.append(run)
    print(f"{len(run_events)} samples extracted")

X = np.array(X_list)
y = np.array(y_list)
run_labels = np.array(run_labels)
y_encoded = np.array([REGION_TO_LABEL[region] for region in y])

print(f"\nFeature matrix shape: {X.shape}")
print(f"Labels shape: {y_encoded.shape}")
print(f"Number of unique regions: {len(np.unique(y_encoded))}")
print(f"\nLabel distribution:")
for region, label in REGION_TO_LABEL.items():
    count = np.sum(y_encoded == label)
    print(f"  {region} (Class {label}): {count} samples")

In [ ]:
top15_csv = TRAIN_RESULTS_DIR / "all_experiments.csv"
df_all = pd.read_csv(top15_csv)
df_sorted = df_all.sort_values('mean_accuracy', ascending=False)
top15 = df_sorted.head(15)

print("="*80)
print("TOP 15 MODELS - Testing on sub-p0002")
print("="*80)
print(f"Models trained on: sub-p0001")
print(f"Testing on: {subject}")
print(f"Total testing samples: {len(y_encoded)}")
print(f"Number of models to test: {len(top15)}")
print("="*80)

In [ ]:
test_results = []

for rank, (idx, model_info) in enumerate(top15.iterrows(), 1):
    print(f"\n{'='*80}")
    print(f"TESTING MODEL {rank}/15 - Experiment {model_info['experiment_id']}")
    print(f"{'='*80}")
    print(f"Architecture: {model_info['architecture']}")
    print(f"Hidden layers: {model_info['hidden1']}" + (f" -> {model_info['hidden2']}" if model_info['hidden2'] != 'None' else ""))
    print(f"Weight decay: {model_info['weight_decay']}, Dropout: {model_info['dropout_rate']}")
    pca_components = None if model_info['pca_components'] == 'None' else int(model_info['pca_components'])
    if pca_components:
        print(f"PCA components: {pca_components}")
    print(f"Training accuracy (sub-p0001): {model_info['mean_accuracy']*100:.2f}% +/- {model_info['std_accuracy']*100:.2f}%")
    
    hidden1 = int(model_info['hidden1'])
    hidden2 = None if model_info['hidden2'] == 'None' else int(model_info['hidden2'])
    dropout_rate = float(model_info['dropout_rate'])
    
    model_path = TRAIN_RESULTS_DIR / model_info['model_file']
    
    fold_accuracies = []
    fold_predictions = []
    fold_true_labels = []
    
    for test_run in range(1, n_runs + 1):
        print(f"  Fold {test_run}/4...", end=' ')
        
        train_mask = run_labels != test_run
        test_mask = run_labels == test_run
        
        X_train = X[train_mask]
        X_test = X[test_mask]
        y_train = y_encoded[train_mask]
        y_test = y_encoded[test_mask]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        if pca_components is not None:
            pca = PCA(n_components=pca_components, random_state=RANDOM_SEED)
            X_train_processed = pca.fit_transform(X_train_scaled)
            X_test_processed = pca.transform(X_test_scaled)
            current_input_size = pca_components
        else:
            X_train_processed = X_train_scaled
            X_test_processed = X_test_scaled
            current_input_size = X.shape[1]
        
        X_test_tensor = FloatTensor(X_test_processed)
        
        model = SomatotopicANN(current_input_size, hidden1, hidden2, 4, dropout_rate)
        model.load_state_dict(LongTensor.load(model_path))
        model.eval()
        
        with no_grad():
            test_outputs = model(X_test_tensor)
            _, test_predictions = max(test_outputs.data, 1)
        
        y_pred = test_predictions.numpy()
        fold_acc = accuracy_score(y_test, y_pred)
        fold_accuracies.append(fold_acc)
        fold_predictions.extend(y_pred)
        fold_true_labels.extend(y_test)
        
        print(f"Acc: {fold_acc*100:.2f}%")
    
    y_pred_all = np.array(fold_predictions)
    y_true_all = np.array(fold_true_labels)
    
    mean_acc = np.mean(fold_accuracies)
    std_acc = np.std(fold_accuracies)
    
    cm = confusion_matrix(y_true_all, y_pred_all)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true_all, y_pred_all, average=None, zero_division=0
    )
    specificities = [calculate_specificity(y_true_all, y_pred_all, c) for c in range(4)]
    
    macro_precision = np.mean(precision)
    macro_sensitivity = np.mean(recall)
    macro_specificity = np.mean(specificities)
    macro_f1 = np.mean(f1)
    
    weighted_precision = np.average(precision, weights=support)
    weighted_sensitivity = np.average(recall, weights=support)
    weighted_specificity = np.average(specificities, weights=support)
    weighted_f1 = np.average(f1, weights=support)
    
    print(f"\n  Testing accuracy (sub-p0002): {mean_acc*100:.2f}% +/- {std_acc*100:.2f}%")
    print(f"  Macro Sensitivity: {macro_sensitivity:.4f}, Macro Specificity: {macro_specificity:.4f}")
    print(f"  Macro Precision: {macro_precision:.4f}, Macro F1: {macro_f1:.4f}")
    
    test_results.append({
        'rank': rank,
        'experiment_id': model_info['experiment_id'],
        'architecture': model_info['architecture'],
        'hidden1': hidden1,
        'hidden2': hidden2 if hidden2 is not None else 'None',
        'weight_decay': model_info['weight_decay'],
        'dropout_rate': dropout_rate,
        'pca_components': pca_components if pca_components is not None else 'None',
        'train_accuracy_sub-p0001': model_info['mean_accuracy'],
        'train_std_sub-p0001': model_info['std_accuracy'],
        'test_accuracy_sub-p0002': mean_acc,
        'test_std_sub-p0002': std_acc,
        'macro_sensitivity': macro_sensitivity,
        'macro_specificity': macro_specificity,
        'macro_precision': macro_precision,
        'macro_f1': macro_f1,
        'weighted_sensitivity': weighted_sensitivity,
        'weighted_specificity': weighted_specificity,
        'weighted_precision': weighted_precision,
        'weighted_f1': weighted_f1,
        'per_class_sensitivity': str(recall.tolist()),
        'per_class_specificity': str(np.array(specificities).tolist()),
        'per_class_precision': str(precision.tolist()),
        'per_class_f1': str(f1.tolist()),
        'model_file': model_info['model_file']
    })

print(f"\n{'='*80}")
print("ALL MODELS TESTED")
print(f"{'='*80}")

test_df = pd.DataFrame(test_results)
output_csv = TEST_RESULTS_DIR / "top15_tested_on_sub-p0002.csv"
test_df.to_csv(output_csv, index=False)
print(f"\nResults saved to: {output_csv}")

In [ ]:
print("\n" + "="*100)
print("CROSS-SUBJECT GENERALIZATION ANALYSIS")
print("="*100)

test_df_sorted = test_df.sort_values('test_accuracy_sub-p0002', ascending=False)

print(f"\nBest performing model on sub-p0002:")
best_model = test_df_sorted.iloc[0]
print(f"  Experiment {best_model['experiment_id']} (Original Rank: #{best_model['rank']})")
print(f"  Architecture: {best_model['architecture']}")
print(f"  Training accuracy (sub-p0001): {best_model['train_accuracy_sub-p0001']*100:.2f}%")
print(f"  Testing accuracy (sub-p0002): {best_model['test_accuracy_sub-p0002']*100:.2f}%")
print(f"  Generalization gap: {(best_model['train_accuracy_sub-p0001'] - best_model['test_accuracy_sub-p0002'])*100:.2f}%")

print(f"\nOverall statistics:")
print(f"  Mean testing accuracy: {test_df['test_accuracy_sub-p0002'].mean()*100:.2f}%")
print(f"  Std testing accuracy: {test_df['test_accuracy_sub-p0002'].std()*100:.2f}%")
print(f"  Best testing accuracy: {test_df['test_accuracy_sub-p0002'].max()*100:.2f}%")
print(f"  Worst testing accuracy: {test_df['test_accuracy_sub-p0002'].min()*100:.2f}%")

avg_generalization_gap = (test_df['train_accuracy_sub-p0001'] - test_df['test_accuracy_sub-p0002']).mean() * 100
print(f"\nAverage generalization gap: {avg_generalization_gap:.2f}%")

print("\n" + "="*100)
print("TOP 5 MODELS BY TESTING PERFORMANCE ON SUB-P0002")
print("="*100)

for i, (_, model) in enumerate(test_df_sorted.head(5).iterrows(), 1):
    print(f"\n#{i} - Experiment {model['experiment_id']} (Original Rank: #{model['rank']})")
    print(f"  Architecture: {model['architecture']}")
    print(f"  Hidden Layers: {model['hidden1']}" + (f" -> {model['hidden2']}" if model['hidden2'] != 'None' else ""))
    pca_str = f", PCA: {model['pca_components']}" if model['pca_components'] != 'None' else ""
    print(f"  Hyperparameters: Weight Decay={model['weight_decay']}, Dropout={model['dropout_rate']}{pca_str}")
    print(f"  Training (sub-p0001): {model['train_accuracy_sub-p0001']*100:.2f}% +/- {model['train_std_sub-p0001']*100:.2f}%")
    print(f"  Testing (sub-p0002): {model['test_accuracy_sub-p0002']*100:.2f}% +/- {model['test_std_sub-p0002']*100:.2f}%")
    print(f"  Generalization gap: {(model['train_accuracy_sub-p0001'] - model['test_accuracy_sub-p0002'])*100:.2f}%")
    print(f"  Macro Sensitivity: {model['macro_sensitivity']:.4f}, Macro Specificity: {model['macro_specificity']:.4f}")

print("\n" + "="*100)